# The perfectns validation problems

The `ultranest.perfectns` module provides likelihood/prior pairs with
**analytic evidence values** (`analytic_lnZ`), designed for validating
nested samplers: because the correct `lnZ` is known, any sampler can be
scored against it.

This notebook shows all five problems, visualises their shape in 2d, and
tabulates the analytic `lnZ` values across dimensionalities.

The problems plug into any nested sampler through their `transform`
(uniform prior on $[0,1]^N$) and `loglike` functions; the pipeline in
`evaluate/perfectns/` drives them in batches.

## The problems

* **NonIsotropicGaussian**
    * A Gaussian, but not equal across axes: the first sigma is large, the last one is orders of magnitudes smaller.
    * $\ln Z \approx 0$, by normalization, when the cube contains the bulk of the gaussian.
* **UncorrelatedFunnel**
    * Neal's funnel, but in non-centered parameterization
    * the first dimension scales the width of all others (a hierarchical model like $\mu \sim \mathcal{N}(0,
  \sigma)$ with per-point widths).
    * Truncated to the cube; its evidence underflows ($-\infty$) at high dimensionality.
* **BoxGaussian**
    * a constant (uniform) very high likelihood inside a small box on top of a shallow, unimportant Gaussian
    * the evidence is the box volume times its height, so `lnZ` is large and positive.
* **Banana**
    * Rosenbrock-like
    * The analytic value is a 2d numerical integration, implemented for $d=2$ only.
* **HyperPyramid**
    * hyper-pyramid (manhattan distance-based likelihood)
    * uniform inside a simplex-like shape that tightens with each dimension

These all have flat prior on $[0,1]^N$ unless stated otherwise.


## Shape of the problems in 2d

Each panel shows the log-likelihood over the unit square for $d=2$ (the
prior is the flat unit cube, so this is the full picture).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from ultranest.perfectns import (
    BananaGaussianLRPS, BoxGaussianLRPS, HyperPyramidLRPS,
    NonIsotropicGaussianLRPS, UncorrelatedFunnelLRPS)

problems = [
    ('NonIsotropicGaussian',
     lambda d: NonIsotropicGaussianLRPS(d)),
    ('UncorrelatedFunnel',
     lambda d: UncorrelatedFunnelLRPS(d, trunc_sigmas=6.0)),
    ('BoxGaussian', lambda d: BoxGaussianLRPS(d)),
    ('Banana', lambda d: BananaGaussianLRPS.standard_problem(d)),
    ('HyperPyramid', lambda d: HyperPyramidLRPS(d)),
]

grid = np.linspace(1e-6, 1 - 1e-6, 501)
X, Y = np.meshgrid(grid, grid)
pts = np.stack([X.flatten(), Y.flatten()], axis=1)

for (name, make) in problems:
    fig, ax = plt.subplots(figsize=(6, 6))
    L = make(2).loglike(pts).reshape(X.shape)
    vmax = np.percentile(L[np.isfinite(L)], 99)
    ax.contourf(X, Y, np.clip(L, vmax - 10, L.max()), levels=50)
    ax.set_title(name)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    #axes.flatten()[-1].axis('off')
    plt.tight_layout()


The gaussian problems are smooth ellipsoids (the NonIsotropicGaussian
strongly anisotropic, the funnel with a curved width profile), the box is
flat with sharp edges, the banana is curved, and the pyramid is
piecewise-linear.

## Analytic lnZ values across dimensionality

`analytic_lnZ()` gives the reference value a nested sampler should
recover. Notes:

* the **banana** value is a 2d numerical integration and raises for
  $d \neq 2$ (marked `NaN`);
* the **funnel** evidence underflows to $-\infty$ from $d=30$ on: the
  truncated likelihood vanishes everywhere in the cube at those widths;
* the NonIsotropicGaussian is normalized and its bulk lies inside the
  cube, so its `lnZ` stays at $\approx 0$; the box concentrates its
  probability mass, so its `lnZ` is large and positive.

In [ ]:
import pandas as pd

dims = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 40]
rows = {}
for name, make in problems:
    vals = []
    for d in dims:
        try:
            vals.append(float(make(d).analytic_lnZ()))
        except Exception:
            vals.append(np.nan)
    rows[name] = vals

df = pd.DataFrame.from_dict(rows, orient='index', columns=dims)
df.index.name = 'problem'
df.round(3)


## Using the problems

Each problem exposes `paramnames`, `transform(u)` and
`loglike(theta)` (vectorized), plus `analytic_lnZ()` and
`make_volume_function()`. They are the standard way to test a nested
sampler end to end: run the sampler and compare its `lnZ` against the
table above.